# Academia–Practice Interaction Mapping Using NLP  
**Notebook 09: Merge and Organize Classified Entities**

**Author:** Kamila Lewandowska  
**Project Phase:** In Progress  
**Last Updated:** July 2025  

---

## Objective

Consolidate and organize outputs from multiple entity classification steps into a unified dataset for further analysis.

---

## Workflow Summary

- Load classified outputs from different NER pipelines (Davlan/XLM-RoBERTa, Stanza, etc.)
- Concatenate all classification tables into a single DataFrame  

---

## Key Outcomes

- **Total classified entity entries after merge and dropping duplicate ID + ORG_Entity pairs:** *20388*  

In [2]:
import pandas as pd


## Merge three tables with annotated entities: common entities and Stanza only (rule-matched and annotated)

In [3]:
# Load both tables
classified_common = pd.read_csv("../output/common_classified_with_ids.csv")
annotated_stanza = pd.read_excel("../output/stanza_non_academic_annotated.xlsx")
rule_matched_stanza = pd.read_csv("../output/stanza_non_academic_rule_matched.csv")
annotated_davlan = pd.read_excel("../output/davlan_non_academic_annotated.xlsx")
rule_matched_davlan = pd.read_csv("../output/davlan_non_academic_rule_matched.csv")

# Rename column in df_annotated to match target column name
annotated_stanza = annotated_stanza.rename(columns={'Annotated_Category': 'Matched_Category', 'Entity_ID': 'ICS_ID'})

# Reorder columns to match final structure
classified_common_final = classified_common[['ICS_ID', 'ORG_Entity', 'Matched_Category']]
annotated_stanza_final = annotated_stanza[['ICS_ID', 'ORG_Entity', 'Matched_Category']]
rule_matched_stanza_final = rule_matched_stanza[['ICS_ID', 'ORG_Entity', 'Matched_Category']]
annotated_davan_final = annotated_davlan[['ICS_ID', 'ORG_Entity', 'Matched_Category']]
rule_matched_davlan_final = rule_matched_davlan[['ICS_ID', 'ORG_Entity', 'Matched_Category']]

# Keep only the desired columns
entities_with_cats_dupl = pd.concat([classified_common_final, annotated_stanza_final, rule_matched_stanza_final, annotated_davan_final, rule_matched_davlan_final], ignore_index=True)

len(entities_with_cats_dupl)

22194

## Data cleaning 

In [4]:
# Drop rows with duplicate ICS_ID and ORG_Entity pairs

entities_with_cats = entities_with_cats_dupl.drop_duplicates(subset=['ICS_ID', 'ORG_Entity']).copy()

len(entities_with_cats)

20388

In [5]:
# Check frequencies of matched categories

entities_with_cats["Matched_Category"].value_counts()

Matched_Category
Other / Unclear                       5090
Government / Public Administration    4689
Company / Business                    3774
International Organization / EU       1473
NGO / Association / Foundation        1471
Cultural Institution / Arts           1119
Media / Publishing                    1057
Health / Hospitals / Medical           590
Education (non-university)             539
Military / Defense / Security          369
Religious Organization                 213
Local Government                         1
Research Institute / Academic            1
University                               1
Name: count, dtype: int64

In [6]:
# Clean irrelevant categories

cat_clean = {
    "University": "Other / Unclear",
    "Research Institute / Academic": "Other / Unclear",
    "Local Government": "Government / Public Administration"
}

entities_with_cats["Matched_Category"]= entities_with_cats["Matched_Category"].replace(cat_clean)

In [7]:
# Check for null values

entities_with_cats["Matched_Category"].isna().sum()

# Replace null values with Other / Unclear category

entities_with_cats["Matched_Category"] = entities_with_cats["Matched_Category"].fillna("Other / Unclear")
entities_with_cats["Matched_Category"].unique()

array(['Company / Business', 'Government / Public Administration',
       'Other / Unclear', 'Media / Publishing',
       'NGO / Association / Foundation',
       'International Organization / EU', 'Cultural Institution / Arts',
       'Education (non-university)', 'Health / Hospitals / Medical',
       'Military / Defense / Security', 'Religious Organization'],
      dtype=object)

## Data organization and exploration

In [8]:
# Check the number of matched categories excluding "Other / Unclear"

entities_with_cats["Matched_Category"].value_counts().sum()

20388

In [16]:
# Sort concatenated entities_with_cats dataframe so that rows with the same ID are grouped together

entities_with_cats = entities_with_cats.sort_values(by="ICS_ID").reset_index(drop=True)

In [17]:
# Check the number of case studies from which entities were extracted

entities_with_cats["ICS_ID"].nunique()

2522

In [19]:
# Export file

entities_with_cats.to_csv("../output/entities_with_cats.csv", index=False)
